In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

patients = spark.table("healthcare_demo.bronze.patients")
referrals = spark.table("healthcare_demo.bronze.referrals")
appointments = spark.table("healthcare_demo.bronze.appointments")
capacity = spark.table("healthcare_demo.bronze.capacity")

In [0]:
patients_clean = (
    patients
    .filter(F.col("patient_id").isNotNull())
    .dropDuplicates(["patient_id"])
    .withColumn("area_code", F.upper(F.trim(F.col("area_code"))))
)

display(patients_clean)

patient_id,age_band,area_code,_ingested_at
P0001,18-30,NG1,2026-11-09T21:03:07.592Z
P0002,46-60,NG2,2026-11-09T21:03:07.592Z
P0003,31-45,NG2,2026-11-09T21:03:07.592Z
P0004,18-30,NG1,2026-11-09T21:03:07.592Z
P0005,61-75,NG1,2026-11-09T21:03:07.592Z
P0006,18-30,NG1,2026-11-09T21:03:07.592Z
P0007,31-45,NG2,2026-11-09T21:03:07.592Z
P0008,18-30,NG2,2026-11-09T21:03:07.592Z
P0009,61-75,NG2,2026-11-09T21:03:07.592Z
P0010,61-75,NG5,2026-11-09T21:03:07.592Z


In [0]:
referrals_clean = (
    referrals
    .filter(F.col("referral_id").isNotNull())
    .withColumn("referral_date", F.to_date("referral_date"))
    .withColumn("specialty", F.initcap(F.trim(F.col("specialty"))))
    .withColumn("priority", F.initcap(F.trim(F.col("priority"))))
    .dropDuplicates(["referral_id"])
)

display(referrals_clean)

referral_id,patient_id,referral_date,specialty,priority,referral_status,_ingested_at
R0001,P0001,2026-10-23,Dermatology,Routine,Open,2026-11-09T21:03:12.531Z
R0002,P0002,2026-10-12,Cardiology,Routine,Open,2026-11-09T21:03:12.531Z
R0003,P0003,2026-07-23,Orthopaedics,Urgent,Open,2026-11-09T21:03:12.531Z
R0004,P0004,2026-09-11,Orthopaedics,Routine,Open,2026-11-09T21:03:12.531Z
R0005,P0005,2026-09-23,Orthopaedics,Routine,Open,2026-11-09T21:03:12.531Z
R0006,P0006,2026-07-28,Orthopaedics,Urgent,Open,2026-11-09T21:03:12.531Z
R0007,P0007,2026-08-23,Orthopaedics,Urgent,Open,2026-11-09T21:03:12.531Z
R0008,P0008,2026-09-26,Cardiology,Urgent,Open,2026-11-09T21:03:12.531Z
R0009,P0009,2026-09-10,Cardiology,Routine,Open,2026-11-09T21:03:12.531Z
R0010,P0010,2026-08-16,Cardiology,Routine,Open,2026-11-09T21:03:12.531Z


In [0]:
appointments_clean = (
    appointments
    .filter(F.col("appointment_id").isNotNull())
    .withColumn("appointment_date", F.to_date("appointment_date"))
    .withColumn("appointment_status", F.initcap(F.trim(F.col("appointment_status"))))
    .dropDuplicates(["appointment_id"])
)

display(appointments_clean)

appointment_id,referral_id,appointment_date,location_id,appointment_status,_ingested_at
A0001,R0001,2026-12-27,L002,Attended,2026-11-09T21:03:15.516Z
A0002,R0002,2026-11-26,L001,Cancelled,2026-11-09T21:03:15.516Z
A0003,R0003,2026-11-27,L003,Attended,2026-11-09T21:03:15.516Z
A0004,R0004,2026-12-20,L001,Attended,2026-11-09T21:03:15.516Z
A0005,R0005,2026-12-08,L001,Cancelled,2026-11-09T21:03:15.516Z
A0006,R0006,2026-11-28,L001,Attended,2026-11-09T21:03:15.516Z
A0007,R0007,2026-12-01,L002,Cancelled,2026-11-09T21:03:15.516Z
A0008,R0008,2026-11-15,L001,Booked,2026-11-09T21:03:15.516Z
A0009,R0009,2026-11-24,L002,Cancelled,2026-11-09T21:03:15.516Z
A0010,R0010,2026-11-19,L003,Booked,2026-11-09T21:03:15.516Z


In [0]:
capacity_clean = (
    capacity
    .filter(F.col("location_id").isNotNull())
    .withColumn("specialty", F.initcap(F.trim(F.col("specialty"))))
    .withColumn("available_slots", F.col("available_slots").cast("int"))
    .dropDuplicates(["location_id", "specialty"])
)

display(capacity_clean)

location_id,location_name,specialty,available_slots,_ingested_at
L001,Central Clinic,Cardiology,5,2026-11-09T21:03:18.992Z
L002,North Clinic,Orthopaedics,8,2026-11-09T21:03:18.992Z
L003,South Clinic,Dermatology,6,2026-11-09T21:03:18.992Z


In [0]:
# Remove duplicate ingestion metadata before joining
patients_clean = patients_clean.drop("_ingested_at")
referrals_clean = referrals_clean.drop("_ingested_at")
appointments_clean = appointments_clean.drop("_ingested_at")
capacity_clean = capacity_clean.drop("_ingested_at")

# Join patients and referrals
patient_referrals = patients_clean.join(
    referrals_clean,
    on="patient_id",
    how="inner"
)

# Join appointments and clinic capacity
silver_data = (
    patient_referrals
    .join(appointments_clean, on="referral_id", how="left")
    .join(
        capacity_clean,
        on=["location_id", "specialty"],
        how="left"
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
)

display(silver_data)

location_id,specialty,referral_id,patient_id,age_band,area_code,referral_date,priority,referral_status,appointment_id,appointment_date,appointment_status,location_name,available_slots,_silver_processed_at
L002,Orthopaedics,R0007,P0007,31-45,NG2,2026-08-23,Urgent,Open,A0007,2026-12-01,Cancelled,North Clinic,8,2026-11-09T21:08:40.798Z
L002,Cardiology,R0009,P0009,61-75,NG2,2026-09-10,Routine,Open,A0009,2026-11-24,Cancelled,null,null,2026-11-09T21:08:40.798Z
L002,Orthopaedics,R0059,P0059,46-60,NG2,2026-07-25,Urgent,Open,A0059,2026-12-21,Booked,North Clinic,8,2026-11-09T21:08:40.798Z
L002,Dermatology,R0061,P0061,31-45,NG1,2026-05-24,Urgent,Open,A0061,2026-11-16,Attended,null,null,2026-11-09T21:08:40.798Z
L003,Orthopaedics,R0072,P0072,31-45,NG5,2026-05-18,Urgent,Open,A0072,2026-12-19,Attended,null,null,2026-11-09T21:08:40.798Z
L002,Dermatology,R0085,P0085,31-45,NG7,2026-09-22,Routine,Open,A0085,2026-11-15,Booked,null,null,2026-11-09T21:08:40.798Z
L002,Cardiology,R0088,P0088,18-30,NG1,2026-09-08,Routine,Open,A0088,2026-12-03,Cancelled,null,null,2026-11-09T21:08:40.798Z
L001,Orthopaedics,R0005,P0005,61-75,NG1,2026-09-23,Routine,Open,A0005,2026-12-08,Cancelled,null,null,2026-11-09T21:08:40.798Z
L001,Cardiology,R0008,P0008,18-30,NG2,2026-09-26,Urgent,Open,A0008,2026-11-15,Booked,Central Clinic,5,2026-11-09T21:08:40.798Z
L002,Orthopaedics,R0011,P0011,18-30,NG2,2026-06-08,Routine,Open,A0011,2026-11-14,Attended,North Clinic,8,2026-11-09T21:08:40.798Z


In [0]:
silver_data = silver_data.withColumn(
    "waiting_days",
    F.datediff(
        F.coalesce(F.col("appointment_date"), F.current_date()),
        F.col("referral_date")
    )
)

display(silver_data)

location_id,specialty,referral_id,patient_id,age_band,area_code,referral_date,priority,referral_status,appointment_id,appointment_date,appointment_status,location_name,available_slots,_silver_processed_at,waiting_days
L002,Orthopaedics,R0007,P0007,31-45,NG2,2026-08-23,Urgent,Open,A0007,2026-12-01,Cancelled,North Clinic,8,2026-11-09T21:09:04.829Z,100
L002,Cardiology,R0009,P0009,61-75,NG2,2026-09-10,Routine,Open,A0009,2026-11-24,Cancelled,null,null,2026-11-09T21:09:04.829Z,75
L002,Orthopaedics,R0059,P0059,46-60,NG2,2026-07-25,Urgent,Open,A0059,2026-12-21,Booked,North Clinic,8,2026-11-09T21:09:04.829Z,149
L002,Dermatology,R0061,P0061,31-45,NG1,2026-05-24,Urgent,Open,A0061,2026-11-16,Attended,null,null,2026-11-09T21:09:04.829Z,176
L003,Orthopaedics,R0072,P0072,31-45,NG5,2026-05-18,Urgent,Open,A0072,2026-12-19,Attended,null,null,2026-11-09T21:09:04.829Z,215
L002,Dermatology,R0085,P0085,31-45,NG7,2026-09-22,Routine,Open,A0085,2026-11-15,Booked,null,null,2026-11-09T21:09:04.829Z,54
L002,Cardiology,R0088,P0088,18-30,NG1,2026-09-08,Routine,Open,A0088,2026-12-03,Cancelled,null,null,2026-11-09T21:09:04.829Z,86
L001,Orthopaedics,R0005,P0005,61-75,NG1,2026-09-23,Routine,Open,A0005,2026-12-08,Cancelled,null,null,2026-11-09T21:09:04.829Z,76
L001,Cardiology,R0008,P0008,18-30,NG2,2026-09-26,Urgent,Open,A0008,2026-11-15,Booked,Central Clinic,5,2026-11-09T21:09:04.829Z,50
L002,Orthopaedics,R0011,P0011,18-30,NG2,2026-06-08,Routine,Open,A0011,2026-11-14,Attended,North Clinic,8,2026-11-09T21:09:04.829Z,159


In [0]:
silver_data.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_demo.silver.patient_referral_details"
)

In [0]:
%sql
SELECT *
FROM healthcare_demo.silver.patient_referral_details
LIMIT 20;

location_id,specialty,referral_id,patient_id,age_band,area_code,referral_date,priority,referral_status,appointment_id,appointment_date,appointment_status,location_name,available_slots,_silver_processed_at,waiting_days
L002,Orthopaedics,R0007,P0007,31-45,NG2,2026-08-23,Urgent,Open,A0007,2026-12-01,Cancelled,North Clinic,8,2026-11-09T21:09:28.250Z,100
L002,Cardiology,R0009,P0009,61-75,NG2,2026-09-10,Routine,Open,A0009,2026-11-24,Cancelled,null,null,2026-11-09T21:09:28.250Z,75
L002,Orthopaedics,R0059,P0059,46-60,NG2,2026-07-25,Urgent,Open,A0059,2026-12-21,Booked,North Clinic,8,2026-11-09T21:09:28.250Z,149
L002,Dermatology,R0061,P0061,31-45,NG1,2026-05-24,Urgent,Open,A0061,2026-11-16,Attended,null,null,2026-11-09T21:09:28.250Z,176
L003,Orthopaedics,R0072,P0072,31-45,NG5,2026-05-18,Urgent,Open,A0072,2026-12-19,Attended,null,null,2026-11-09T21:09:28.250Z,215
L002,Dermatology,R0085,P0085,31-45,NG7,2026-09-22,Routine,Open,A0085,2026-11-15,Booked,null,null,2026-11-09T21:09:28.250Z,54
L002,Cardiology,R0088,P0088,18-30,NG1,2026-09-08,Routine,Open,A0088,2026-12-03,Cancelled,null,null,2026-11-09T21:09:28.250Z,86
L001,Orthopaedics,R0005,P0005,61-75,NG1,2026-09-23,Routine,Open,A0005,2026-12-08,Cancelled,null,null,2026-11-09T21:09:28.250Z,76
L001,Cardiology,R0008,P0008,18-30,NG2,2026-09-26,Urgent,Open,A0008,2026-11-15,Booked,Central Clinic,5,2026-11-09T21:09:28.250Z,50
L002,Orthopaedics,R0011,P0011,18-30,NG2,2026-06-08,Routine,Open,A0011,2026-11-14,Attended,North Clinic,8,2026-11-09T21:09:28.250Z,159


In [0]:
print("Silver record count:", silver_data.count())

print(
    "Missing patient IDs:",
    silver_data.filter(F.col("patient_id").isNull()).count()
)

print(
    "Missing referral IDs:",
    silver_data.filter(F.col("referral_id").isNull()).count()
)

print(
    "Missing capacity records:",
    silver_data.filter(F.col("available_slots").isNull()).count()
)

Silver record count: 100
Missing patient IDs: 0
Missing referral IDs: 0
Missing capacity records: 55


In [0]:

from pyspark.sql import functions as F

# Read Silver table
silver_df = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

# Define data quality checks
total_records = silver_df.count()

duplicate_referrals = (
    silver_df
    .groupBy("referral_id")
    .count()
    .filter(F.col("referral_id").isNotNull() & (F.col("count") > 1))
    .count()
)

missing_patient_id = silver_df.filter(
    F.col("patient_id").isNull()
).count()

missing_referral_id = silver_df.filter(
    F.col("referral_id").isNull()
).count()

missing_specialty = silver_df.filter(
    F.col("specialty").isNull()
).count()

missing_priority = silver_df.filter(
    F.col("priority").isNull()
).count()

missing_referral_date = silver_df.filter(
    F.col("referral_date").isNull()
).count()

invalid_priority = silver_df.filter(
    ~F.col("priority").isin("Urgent", "Routine") |
    F.col("priority").isNull()
).count()

invalid_specialty = silver_df.filter(
    ~F.col("specialty").isin(
        "Cardiology",
        "Dermatology",
        "Orthopaedics"
    ) |
    F.col("specialty").isNull()
).count()

negative_waiting_days = silver_df.filter(
    F.col("waiting_days") < 0
).count()

missing_waiting_days = silver_df.filter(
    F.col("waiting_days").isNull()
).count()

invalid_appointment_date = silver_df.filter(
    F.col("appointment_date").isNotNull() &
    F.col("referral_date").isNotNull() &
    (F.col("appointment_date") < F.col("referral_date"))
).count()

# Create data quality results
quality_results = [
    ("Total Silver records", total_records, "INFO"),
    ("Duplicate referral IDs", duplicate_referrals, "FAIL"),
    ("Missing patient IDs", missing_patient_id, "FAIL"),
    ("Missing referral IDs", missing_referral_id, "FAIL"),
    ("Missing specialties", missing_specialty, "FAIL"),
    ("Missing priorities", missing_priority, "FAIL"),
    ("Missing referral dates", missing_referral_date, "FAIL"),
    ("Invalid priorities", invalid_priority, "FAIL"),
    ("Invalid specialties", invalid_specialty, "FAIL"),
    ("Negative waiting days", negative_waiting_days, "FAIL"),
    ("Missing waiting days", missing_waiting_days, "FAIL"),
    ("Appointment before referral", invalid_appointment_date, "FAIL")
]

quality_df = spark.createDataFrame(
    quality_results,
    ["quality_check", "failed_records", "check_type"]
)

quality_df = quality_df.withColumn(
    "result",
    F.when(
        (F.col("check_type") == "INFO") |
        (F.col("failed_records") == 0),
        "PASS"
    ).otherwise("FAIL")
)

display(quality_df.orderBy("check_type", "quality_check"))

# Overall data quality status
failed_checks = quality_df.filter(
    (F.col("check_type") == "FAIL") &
    (F.col("result") == "FAIL")
).count()

if failed_checks == 0:
    print("Silver data quality validation PASSED")
else:
    print(f"Silver data quality validation FAILED: {failed_checks} checks need attention")


quality_check,failed_records,check_type,result
Appointment before referral,0,FAIL,PASS
Duplicate referral IDs,0,FAIL,PASS
Invalid priorities,0,FAIL,PASS
Invalid specialties,0,FAIL,PASS
Missing patient IDs,0,FAIL,PASS
Missing priorities,0,FAIL,PASS
Missing referral IDs,0,FAIL,PASS
Missing referral dates,0,FAIL,PASS
Missing specialties,0,FAIL,PASS
Missing waiting days,0,FAIL,PASS


Silver data quality validation PASSED
